# Kaggle Submission

In [5]:
import pandas as pd
import numpy as np
from typing import List, Dict
from sklearn.linear_model import LogisticRegression

### Helper Functions

In [6]:
BINNING_CONFIG = {
    "RevolvingUtilizationOfUnsecuredLines": {
        "bins": [-np.inf, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0, np.inf],
        "labels": ["<10%", "10-30%", "30-50%", "50-70%", "70-90%", "90-100%", ">100%"]
    },
    
    "age": {
        "bins": [-np.inf, 30, 40, 50, 60, 70, np.inf],
        "labels": ["<30", "30-39", "40-49", "50-59", "60-69", "70+"]
    },
    
    "NumberOfTime30-59DaysPastDueNotWorse": {
        "bins": [-np.inf, 1, 2, np.inf],
        "labels": ["0", "1", "2+"]
    },
    
    "DebtRatio": {
        "bins": [-np.inf, 0.2, 0.4, 0.6, 1.0, 5.0, np.inf],
        "labels": ["<20%", "20-40%", "40-60%", "60-100%", "1-5x", ">5x"]
    },
    
    "MonthlyIncome": {
        "bins": [-np.inf, 2500, 5000, 7500, 10000, 15000, np.inf],
        "labels": ["<2.5k", "2.5k-5k", "5k-7.5k", "7.5k-10k", "10k-15k", ">15k"]
    },
    
    "NumberOfOpenCreditLinesAndLoans": {
        "bins": [-np.inf, 3, 6, 10, 15, np.inf],
        "labels": ["0-2", "3-5", "6-9", "10-14", "15+"]
    },
    
    "NumberOfTimes90DaysLate": {
        "bins": [-np.inf, 1, 2, np.inf],
        "labels": ["0", "1", "2+"]
    },
    
    "NumberRealEstateLoansOrLines": {
        "bins": [-np.inf, 1, 2, 3, np.inf],
        "labels": ["0", "1", "2", "3+"]
    },
    
    "NumberOfTime60-89DaysPastDueNotWorse": {
        "bins": [-np.inf, 1, 2, np.inf],
        "labels": ["0", "1", "2+"]
    },
    
    "NumberOfDependents": {
        "bins": [-np.inf, 1, 2, 3, np.inf],
        "labels": ["0", "1", "2", "3+"]
    }
}

In [7]:
def bin_features(series: pd.Series, bins: List, labels: List[str], right: bool = False, missing_label = "Missing"):
    group = pd.cut(series, bins = bins, labels = labels, right = right, include_lowest = True)

    group = group.cat.add_categories(missing_label).fillna(missing_label)

    return group.cat.remove_unused_categories()

In [8]:
def calculate_woe(df: pd.DataFrame, feature: str, target: str = "SeriousDlqin2yrs", epilson: float = 1e-6) -> tuple[pd.DataFrame, float]:
    total_goods = (df[target] == 0).sum()
    total_bads = (df[target] == 1).sum()

    summary = df.groupby(feature, observed = False)[target].agg(
        total = "count",
        bads = "sum"
    ).reset_index()

    summary["goods"] = summary["total"] - summary["bads"]
    summary["bad_rate"] = (summary["bads"] / summary["total"] * 100).round(2)

    summary["percentage_good"] = summary["goods"] / total_goods
    summary["percentage_bad"] = summary["bads"] / total_bads

    summary["percentage_good"] = summary["percentage_good"].replace(0, epilson)
    summary["percentage_bad"] = summary["percentage_bad"].replace(0, epilson)

    summary["WoE"] = np.log(summary["percentage_good"] / summary["percentage_bad"])
    summary["IV"] = (summary["percentage_good"] - summary["percentage_bad"]) * summary["WoE"]

    total_iv = summary["IV"].sum()
    
    return summary, total_iv

In [9]:
def transform_woe(df, binning_config, woe_map):
    df_woe = pd.DataFrame(index = df.index)

    for feature, config in binning_config.items():
        bins = config["bins"]
        labels = config["labels"]

        group = bin_features(df[feature], bins, labels)
        group_name = f"{feature}_group"

        feature_woe_lookup = {
            k: v["WoE"] for k, v in woe_map[group_name].items()
        }

        df_woe[f"{feature}_group"] = (group.astype(str).map(feature_woe_lookup)).astype(float)

    return df_woe

In [17]:
def woe_mapping(X, y, config: Dict = BINNING_CONFIG) -> tuple[Dict, Dict]:
    woe_maps = {}
    IV_map = {}
    for feature, config in BINNING_CONFIG.items():
        bins = config["bins"]
        labels = config["labels"]
        
        group = bin_features(X[feature], bins, labels)
        group_name = f"{feature}_group"
        X[group_name] = group

        df_concat = pd.concat([X, y], axis = 1)

        summary, total_iv = calculate_woe(df_concat, group_name)

        woe_maps[group_name] = {
            row[group_name]: {
                "WoE": row["WoE"],
                "IV": row["IV"]
            }for _, row in summary.iterrows()
        }

        IV_map[group_name] = total_iv
    return woe_maps, IV_map


### Preparing Data

In [22]:
train_dataset_path = "../data/raw/cs-training.csv"

df = pd.read_csv(train_dataset_path)
if "Unnamed: 0" in df.columns:
    df = df.drop(columns = ["Unnamed: 0"])

print(f"Total Rows: {df.shape[0]}, Total Columns: {df.shape[1]}")
df.head()

Total Rows: 150000, Total Columns: 11


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [11]:
df = df[df["age"] > 0]
print(f"Total Rows: {df.shape[0]}, Total Columns: {df.shape[1]}")

Total Rows: 149999, Total Columns: 11


In [15]:
X = df.drop(columns = ["SeriousDlqin2yrs"])
y = df["SeriousDlqin2yrs"]

print(f"X -> Total Rows: {X.shape[0]}, Total Coulmns: {X.shape[1]}")
print(f"y -> Total Rows: {y.shape[0]}")

X -> Total Rows: 149999, Total Coulmns: 10
y -> Total Rows: 149999


In [18]:
woe_map, IV_mapping = woe_mapping(X, y)
print(woe_map)

{'RevolvingUtilizationOfUnsecuredLines_group': {'<10%': {'WoE': 1.3570584703336197, 'IV': 0.45527887338696665}, '10-30%': {'WoE': 0.7918633736053746, 'IV': 0.08535616414812597}, '30-50%': {'WoE': 0.14224284960943961, 'IV': 0.0020081580203873887}, '50-70%': {'WoE': -0.39008242349388433, 'IV': 0.013635136870616337}, '70-90%': {'WoE': -0.8899469542063498, 'IV': 0.07673959979140667}, '90-100%': {'WoE': -1.2121133405265114, 'IV': 0.27565665855343185}, '>100%': {'WoE': -2.1116775823284923, 'IV': 0.22974358627192157}}, 'age_group': {'<30': {'WoE': -0.618470449457176, 'IV': 0.029447684180218498}, '30-39': {'WoE': -0.44702169876417674, 'IV': 0.03752812601924508}, '40-49': {'WoE': -0.243407637195056, 'IV': 0.015095509648552408}, '50-59': {'WoE': 0.03763859534207085, 'IV': 0.0003280134003953688}, '60-69': {'WoE': 0.6419544912775415, 'IV': 0.06052023195864887}, '70+': {'WoE': 1.1047295803502375, 'IV': 0.10008057258139372}}, 'NumberOfTime30-59DaysPastDueNotWorse_group': {'0': {'WoE': 0.541728141757

In [21]:
X_train_woe = transform_woe(X, BINNING_CONFIG, woe_map)
print(f"Number of NaN values:\n{X_train_woe.isnull().sum()}")
X_train_woe.head(3)

Number of NaN values:
RevolvingUtilizationOfUnsecuredLines_group    0
age_group                                     0
NumberOfTime30-59DaysPastDueNotWorse_group    0
DebtRatio_group                               0
MonthlyIncome_group                           0
NumberOfOpenCreditLinesAndLoans_group         0
NumberOfTimes90DaysLate_group                 0
NumberRealEstateLoansOrLines_group            0
NumberOfTime60-89DaysPastDueNotWorse_group    0
NumberOfDependents_group                      0
dtype: int64


,RevolvingUtilizationOfUnsecuredLines_group,age_group,NumberOfTime30-59DaysPastDueNotWorse_group,DebtRatio_group,MonthlyIncome_group,NumberOfOpenCreditLinesAndLoans_group,NumberOfTimes90DaysLate_group,NumberRealEstateLoansOrLines_group,NumberOfTime60-89DaysPastDueNotWorse_group,NumberOfDependents_group
0,-0.889947,-0.243408,-1.901119,-0.508928,0.271607,0.122237,0.389724,-0.253622,0.288207,-0.209356
1,-1.212113,-0.243408,0.541728,0.095820,-0.292943,0.000814,0.389724,-0.235963,0.288207,-0.102571
2,-0.390082,-0.447022,-0.903720,0.095820,-0.292943,-0.727830,-1.957961,-0.235963,0.288207,0.139835


In [28]:
test_dataset_path = "../data/raw/cs-test.csv"
X_test = pd.read_csv(test_dataset_path)

if "Unnamed: 0" in X_test.columns or "SeriousDlqin2yrs" in X_test.columns:
    X_test = X_test.drop(columns = ["Unnamed: 0", "SeriousDlqin2yrs"])

X_test.head()

,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,0.885519,43,0,0.177513,5700.0,4,0,0,0,0.0
1,0.463295,57,0,0.527237,9141.0,15,0,4,0,2.0
2,0.043275,59,0,0.687648,5083.0,12,0,1,0,2.0
3,0.280308,38,1,0.925961,3200.0,7,0,2,0,0.0
4,1.000000,27,0,0.019917,3865.0,4,0,0,0,1.0


In [29]:
X_test_woe = transform_woe(X_test, BINNING_CONFIG, woe_map)
print(f"Number of NaN values:\n{X_test_woe.isnull().sum()}")
X_test_woe.head(3)

Number of NaN values:
RevolvingUtilizationOfUnsecuredLines_group    0
age_group                                     0
NumberOfTime30-59DaysPastDueNotWorse_group    0
DebtRatio_group                               0
MonthlyIncome_group                           0
NumberOfOpenCreditLinesAndLoans_group         0
NumberOfTimes90DaysLate_group                 0
NumberRealEstateLoansOrLines_group            0
NumberOfTime60-89DaysPastDueNotWorse_group    0
NumberOfDependents_group                      0
dtype: int64


,RevolvingUtilizationOfUnsecuredLines_group,age_group,NumberOfTime30-59DaysPastDueNotWorse_group,DebtRatio_group,MonthlyIncome_group,NumberOfOpenCreditLinesAndLoans_group,NumberOfTimes90DaysLate_group,NumberRealEstateLoansOrLines_group,NumberOfTime60-89DaysPastDueNotWorse_group,NumberOfDependents_group
0,-0.889947,-0.243408,0.541728,0.095820,0.031806,0.000814,0.389724,-0.235963,0.288207,0.139835
1,0.142243,0.037639,0.541728,-0.151462,0.271607,-0.061849,0.389724,-0.253622,0.288207,-0.209356
2,1.357058,0.037639,0.541728,-0.508928,0.031806,0.122237,0.389724,0.256648,0.288207,-0.209356


### Training and Testing

In [33]:
y_good = 1 - y
y_good.shape

(149999,)

In [34]:
clf = LogisticRegression(penalty = None, solver = "lbfgs", max_iter = 1000).fit(X_train_woe, y_good)

/media/divyanshu/MyStorage1/Learning/01_Credit_Fraud/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


In [35]:
probability = clf.predict_proba(X_test_woe)[:,0]

In [36]:
print(probability)

[0.07211484 0.047765   0.02420575 ... 0.00965868 0.06133374 0.0453517 ]


In [53]:
X_test = pd.read_csv(test_dataset_path)
submission = pd.DataFrame({
    "Id": X_test["Unnamed: 0"],
    "Probability": probability
})

print(submission)

            Id  Probability
0            1     0.072115
1            2     0.047765
2            3     0.024206
3            4     0.079420
4            5     0.106230
...        ...          ...
101498  101499     0.031853
101499  101500     0.349425
101500  101501     0.009659
101501  101502     0.061334
101502  101503     0.045352

[101503 rows x 2 columns]


In [54]:
pd.Series(probability).describe()

count    101503.000000
mean          0.066982
std           0.118385
min           0.007296
25%           0.014666
50%           0.025636
75%           0.062768
max           0.946699
dtype: float64

In [55]:
import os
os.makedirs("../submissions", exist_ok = True)

submission_path = "../submissions/01.csv"

submission.to_csv(submission_path, index=False)

print(f"Saved submission to {submission_path} with shape {submission.shape}")

Saved submission to ../submissions/01.csv with shape (101503, 2)
